# Exercise 1 - Equipment costing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/01_equipment_costing.ipynb)

Every techno-economic assessment (TEA) starts from a list of equipment with sizes taken from a process
simulation or a hand calculation. In this notebook we turn those sizes into **purchased** and **installed
(direct) costs** with OpenPyTEA's `Equipment` class, and we build the equipment list of a small green-ammonia
synthesis loop that we will use for the rest of the session.

This is Exercise 1 of 6. Bonus tasks are for participants who finish early; skip them if you are behind.

## Learning goals

- Create an `Equipment` object from a category, type, size parameter, and material.
- Explain how OpenPyTEA computes the purchased cost (cost correlation), adjusts it for inflation (CEPCI), and turns it into a direct cost (installation factors).
- Browse the built-in cost-correlation database to find the right category, type, and size units.
- Use a vendor quote instead of a correlation, change the cost year, and let OpenPyTEA split oversized equipment into parallel units.
- Assemble the equipment list of the workshop's demo plant.

## Background references

- Towler, G., & Sinnott, R. (2022). *Chemical Engineering Design* (3rd ed.). Elsevier. https://doi.org/10.1016/C2019-0-02025-0 (chapter 7: capital cost estimating)
- Turton, R., Shaeiwitz, J.A., Bhattacharyya, D., & Whiting, W.B. (2018). *Analysis, Synthesis, and Design of Chemical Processes* (5th ed.). Prentice Hall. (appendix A: cost correlations)
- OpenPyTEA documentation: https://openpytea.readthedocs.io

## 0) Setup

In [ ]:
# Run this cell first. It installs OpenPyTEA if it is missing (for example on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

## 1) From size to cost in three steps

![OpenPyTEA architecture](https://raw.githubusercontent.com/pbtamarona/OpenPyTEA/main/examples/img/architecture.png)

For each equipment item OpenPyTEA does the following:

1. **Purchased cost** $C_p$ from a cost correlation selected by `category` and `type`. The database
   (`cost_correlations.csv`, shipped with the package) stores 400+ correlations from Towler & Sinnott, Turton et al.,
   Seider et al., Ulrich, Perry, and journal papers, each with its size parameter, validity range, and cost year.
2. **Inflation adjustment** to the target year (2024 by default) with the Chemical Engineering Plant Cost Index:
   $C_{\text{target}} = C_{\text{base}} \cdot \text{CEPCI}_{\text{target}} / \text{CEPCI}_{\text{base}}$.
3. **Direct (installed) cost** with the factorial method of Towler & Sinnott:
   $C_D = C_p \left[(1 + f_p) f_m + (f_{er} + f_{el} + f_i + f_c + f_s + f_l)\right]$,
   where $f_m$ is the material factor and the other factors (piping, erection, electrical, instrumentation,
   civil, structural, lagging) depend on the `process_type` (`Fluids`, `Solids`, `Mixed`, `Electrical`).

Let us cost a shell-and-tube heat exchanger of 250 m² in 304 stainless steel:

In [ ]:
from openpytea import Equipment

exchanger = Equipment(
    name="E-101",
    param=250,                          # heat-transfer area in m2 (the units come from the correlation)
    process_type="Fluids",
    category="Heat Exchangers",
    type="U-tube shell & tube",
    material="304 stainless steel",
)

print(exchanger)
print(f"Purchased cost (2024 USD): {exchanger.purchased_cost:,.0f}")
print(f"Direct cost    (2024 USD): {exchanger.direct_cost:,.0f}")
print(f"Material factor          : {exchanger.material_factor}")

The direct cost is about 3.7 times the purchased cost: this is the well-known "Lang factor" effect for fluid
processes. The installation factors used are class attributes, so you can always inspect (or override) them:

In [ ]:
import pandas as pd

print("Installation factors per process type:")
display(pd.DataFrame(Equipment.process_factors))

print("A few material factors:")
print({m: Equipment.material_factors[m] for m in ["Carbon steel", "304 stainless steel", "316 stainless steel", "Hastelloy C", "Nickel"]})

## 2) Finding the right correlation

The most common question is *"what category, type, and units do I use?"*. The database is a CSV file inside the
package, so we can load it with pandas and filter it. The `units` column tells you what `param` must be.

In [ ]:
from importlib.resources import files

correlations = pd.read_csv(files("openpytea") / "data" / "cost_correlations.csv")
print(f"{len(correlations)} correlations in {correlations['category'].nunique()} categories")

# All categories
print(sorted(correlations["category"].unique()))

In [ ]:
# All correlations for compressors: type, size units, validity range, cost year, and source
columns = ["category", "type", "units", "s_lower", "s_upper", "cost_year", "source"]
correlations[correlations["category"].str.contains("Compressors", case=False)][columns]

> **Tip.** Matching on `category` and `type` is case-insensitive. When several correlations share the same
> category and type, OpenPyTEA takes the first one in the file. Pass `cost_func="<key>"` to pick a specific one.

## 3) Four things you will need in practice

**3a. Material and process type change the installed cost, not the purchased cost**

In [ ]:
for material in ["Carbon steel", "304 stainless steel", "Hastelloy C"]:
    e = Equipment(name="E-101", param=250, process_type="Fluids", category="Heat Exchangers",
                  type="U-tube shell & tube", material=material)
    print(f"{material:<20} purchased {e.purchased_cost:>10,.0f}   direct {e.direct_cost:>10,.0f}")

**3b. A vendor quote or literature value replaces the correlation** (give `purchased_cost` and its `cost_year`, OpenPyTEA still escalates it to the target year)

In [ ]:
electrolyzer_stack = Equipment(
    name="Electrolyzer stack",
    param=None,                         # not needed when the cost is given directly
    process_type="Fluids",
    category="Electrolysis",            # free label: this category is not in the database
    type="Alkaline stack",
    purchased_cost=17_194_000,          # quote in 2020 USD (DOE report)
    cost_year=2020,
)
print(f"2020 quote escalated to 2024: {electrolyzer_stack.purchased_cost:,.0f} USD")

**3c. Costs in another year** (`target_year`) and **oversized equipment** (automatic parallel units when `param` exceeds the correlation's upper bound)

In [ ]:
exchanger_2019 = Equipment(name="E-101", param=250, process_type="Fluids", category="Heat Exchangers",
                           type="U-tube shell & tube", material="304 stainless steel", target_year=2019)
print(f"Same exchanger in 2019 USD: {exchanger_2019.purchased_cost:,.0f} (2024 USD: {exchanger.purchased_cost:,.0f})")

big_compressor = Equipment(name="C-201", param=45_000, process_type="Fluids",
                           category="Compressors, fans, & blowers", type="Compressor, centrifugal")
print(f"45 MW compressor -> {big_compressor.num_units} parallel units, direct cost {big_compressor.direct_cost:,.0f} USD")

**3d. Assemblies** (skim if you are short on time) such as a pressure-swing adsorption (PSA) skid can be built from sub-components with
`CompositeEquipment`. The composite behaves like a single equipment item in the plant.

In [ ]:
from openpytea import CompositeEquipment

vessels = Equipment(name="Adsorber vessels", param=8.0, process_type="Fluids", category="Pressure vessels",
                    type="Vertical", material="Carbon steel", num_units=2)       # two identical vessels of 8 m3
zeolite = Equipment(name="Zeolite 5A", param=2 * 150.0, process_type="Solids",
                    category="Packings & adsorbents", type="Molecular sieves")    # ft3 of adsorbent
psa = CompositeEquipment(name="PSA unit", process_type="Fluids", components=[vessels, zeolite],
                         category="Adsorption", type="Pressure-swing adsorber")
print(psa)
psa.breakdown().round(0)

## 4) Task 1.1: cost the green-ammonia synthesis loop

The demo plant of this workshop is a small **green-ammonia synthesis loop** (100 t NH₃/day) fed with hydrogen
from an electrolyzer and nitrogen from an air separation unit. The process simulation gives the following sizes:

| Tag | Equipment | Category | Type | Size (`param`) | Material |
|---|---|---|---|---|---|
| C-101 | Syngas compressor | `Compressors, fans, & blowers` | `Compressor, centrifugal` | 1 200 kW driver power | Carbon steel |
| M-101 | Compressor motor | `Motors & generators` | `Totally enclosed motor` | 1 200 / 0.90 kW shaft power | default |
| E-101 | Feed/effluent exchanger | `Heat Exchangers` | `U-tube shell & tube` | 250 m² | 304 stainless steel |
| R-101 | Ammonia converter | `Reactors` | `Tubular fixed bed` | 12 000 kg shell mass | 304 stainless steel |
| V-101 | Ammonia separator | `Pressure vessels` | `Vertical` | 25 m³ | Carbon steel |
| K-101 | Refrigeration unit | `Utilities` | `Packaged mechanical refrigerator` | 800 kW evaporator duty | default |
| P-101 | Product pump | `Pumps` | `Single-stage centrifugal` | 20 L/s | Carbon steel |

The motor is an electrical item, so use `process_type="Electrical"` for it; everything else is `"Fluids"`.

**Tasks**
1. Create the seven `Equipment` objects and put them in a list called `equipment_list`.
2. Print a table with the name, purchased cost, and direct cost of each item, and the total direct cost (this total is the ISBL of the plant).
3. Which item dominates? Does that match your engineering intuition?

In [ ]:
# YOUR CODE HERE
# Hints:
#   Equipment(name=..., param=..., process_type=..., category=..., type=..., material=...)
#   Leave `material` out for the motor and the refrigeration unit (the correlation's default material is used).
#   pd.DataFrame([{"name": e.name, "purchased": e.purchased_cost, "direct": e.direct_cost} for e in equipment_list])

## 5) Task 1.2 (bonus): how sensitive is the estimate to your choices?

Using the ammonia converter (R-101), answer the following:

1. What is the direct cost in carbon steel instead of 304 stainless steel? By what factor does it change?
2. What is the purchased cost expressed in 2015 USD instead of 2024 USD?
3. Look up the validity range of the `Tubular fixed bed` correlation in the database. Is 12 000 kg inside it?

In [ ]:
# YOUR CODE HERE

## Recap

- `Equipment(name, param, process_type, category, type, material)` looks up a correlation, escalates the cost with CEPCI, and applies installation factors.
- Check the `units` column of the database before you set `param`; the validity range tells you how far you are extrapolating.
- Vendor quotes go in via `purchased_cost` and `cost_year`; assemblies via `CompositeEquipment`.
- The list of `Equipment` objects is the input to the `Plant` we build in the next notebook.

**Next:** [Exercise 2 - Building a plant](02_building_a_plant.ipynb)